# Finite Difference Methods

Finite Difference Methods (FDM) are one of the popular numerical methods used in computational finance. FDM are discretizations used for solving differential equations by approximating them with difference equations. It is one of the simplest and the oldest methods to solve differential equations. These techniques were applied to numerical applications as early as 1950s.

FDM are similar in approach to the (binomial) tress. However, instead of discretizing asset prices and the passage of time in a tree structure, it discretizes in a grid -  with time and price steps - by calculating the value at every possible grid points.

**Explicit, Implicit and Crank-Nicolson** are the three popular approaches of FDM. The explicit methods are simple to implement, but it does not always converge and largely depends on the size of the time and asset step. Explicit methods are unstable when compared to other two methods. Finite Difference approach is peferred for low dimensional problem, usually upto 4 dimensions.

## Explicit FDM

Explicit Finite Difference Method is a numerical technique used to solve partial differential equations (PDEs) by approximating derivatives with finite differences. It is "explicit" in the sense that the new values in the grid are computed directly from the known values of the previous time step, without needing to solve a system of equations. In Explicit FDM, the solution at the next time step is computed explicitly from the known values at the current time step. This method is straightforward and involves updating each grid point using a formula derived from the PDE and its boundary conditions.

**Note:** Explicit FDM is a forward marching scheme in its numerical computation, but it's applied in a backward-looking manner for option pricing problems. The numerical scheme is forward marching (from known to unknown). The application to option pricing is backward-looking (from expiration to present).

## Section 1: Option Pricing Techniques

As with other option pricing techniques Explicit Finite Difference methods are used to price options using what is essentially a three step process.

**Step 1:** Generate the grid by specifying grid points.<br>
**Step 2:** Specify the final or initial conditions.<br>
**Step 3:** Use boundary conditions to calculate option values and step back down the grid to fill it.

**European Option**

To price an option, we generate a finite grid of a specified asset and time steps for a given maturity. Next, we specify the initial and boundary conditions to calculate payoff when S and T equals zero. We then step back to fill the grid with newer values derived from the earlier values.

**Note:** Certain third-party Python libraries (e.g., yfinance, quantmod) rely on external data sources that may be blocked or inaccessible in China. Those based in China are advised to consult their local tutor or replace these modules with locally permitted alternatives.

**Install Packages**

In [ ]:
!pip install -q quantmod

**Import Libraries**

In [ ]:
# Importing libraries
import pandas as pd
import numpy as np
import quantmod.charts

# Set max row and columns to 300 and 100
pd.set_option('display.max_rows', 300)
pd.set_option('display.max_columns', 100)

**Specify Parameters**

In [ ]:
# Specify the parameters for FDM
T    = 1                              # time to maturity in years
E    = 100                            # strike price
r    = .05                            # riskfree rate
vol  = .20                            # volatility
call = True                           # call = True

NAS  = 20                             # number of asset steps
ds   = 2*E/NAS                        # asset step size

dt   = (0.9/vol**2/NAS**2)            # time step size, for stability - refer lecture
NTS  = int(T/dt) + 1                  # number of time steps
dt   = T/NTS                          # time step size [Expiration as int # of time steps away]

american = False                      # american = False

### Generate Grid

Build the grid with the above input parameters

In [ ]:
# Create asset steps i*ds
s = np.arange(0, (NAS+1)*ds, ds)
s

In [ ]:
# Create time steps k*dt
# k goes from 0,1,...,10;
# t goes from 1,0.9,...0
t= T-np.arange(NTS*dt,-dt,-dt)[::-1]
t

In [ ]:
# Verify the steps size
s.shape, t.shape

In [ ]:
# Initialize the grid with zeros
grid = np.zeros((len(s), len(t)))

# Subsume the grid points into a dataframe
# with asset price as index and time steps as columns
grid = pd.DataFrame(grid, index=s, columns=np.around(t, 3))
grid

### Set up Payoff

Specify payoffs at expiration for calls
$$V_i^{0} = max(i\delta s - E, 0)$$

In [ ]:
# Set Payoff at Expiration
flag = 1 if call else -1
grid.iloc[:,-1] = np.maximum(flag * (s - E), 0)

In [ ]:
# Verify the grid
grid

In [ ]:
# Store payoff for early exercise
p = grid.iloc[:, -1].copy()
p

### Fill the Grid

Specify boundary condition at S = $0$
$$V_0^{k} = (1 - r \delta t) V_0^{k-1}$$

Specify boundary condition at S = $\infty$
$$V_i^{k} = 2V_{i-1}^{k} - V_{i-2}^{k}$$

In [ ]:
# Fill the grid
for k in range(len(t)-2, -1, -1):  # Backwards in time
    for i in range(1, len(s)-1):
        delta = (grid.iloc[i+1, k+1] - grid.iloc[i-1, k+1]) / (2*ds)
        gamma = (grid.iloc[i+1, k+1] - 2*grid.iloc[i, k+1] + grid.iloc[i-1, k+1]) / (ds**2)
        theta = (-0.5 * vol**2 * s[i]**2 * gamma) - (r*s[i]*delta) + (r*grid.iloc[i, k+1])

        # Vnew = Vold - theta * dt
        grid.iloc[i, k] = grid.iloc[i, k+1] - dt*theta

    # Set boundary condition at S = 0
    grid.iloc[0, k] = grid.iloc[0, k+1] * np.exp(-r*dt)

    # Set boundary condition at S = 2E = infinity (some large values)
    grid.iloc[-1, k] = 2*grid.iloc[-2, k] - grid.iloc[-3, k]

    # Check for early exercise
    if american:
        intrinsic_value = np.maximum(flag * (s - E), 0) # Compute intrinsic value at t_k
        grid.iloc[:, k] = np.maximum(grid.iloc[:, k], intrinsic_value)

# round grid values to 2 decimals
grid = np.around(grid,2)

In [ ]:
# Output the option values
grid

In [ ]:
# Print out stock, payoff and option value
data = {
    "Stock": s,
    "Payoff": p,
    "Option": grid.iloc[:,0]
}

option_value_2D = pd.DataFrame(data)
# print(option_value_2D.to_string(index=False))
option_value_2D

In [ ]:
# Plot option value and payoff
option_value_2D[['Payoff', 'Option']].iplot(kind='overlay', title='Payoff & Option Value')

## Section 2: User Defined Function
Let's subsume above grid calculation into a function for ease of use. All we have to do is to combine the above code blocks into a single function.

In [ ]:
# Create function to price options
def efdmoption(E=100, vol=0.20, r=0.05, T=1, NAS=20, call=False, american=False):

    ds   = 2* E / NAS                     # asset step size
    dt   = (0.9/vol**2/NAS**2)            # time step size, for stability
    NTS  = int(T / dt) + 1                # number of time steps
    dt   = T / NTS                        # time step size [Expiration as int # of time steps away]

    # Create asset steps i*ds
    s = np.arange(0, (NAS+1)*ds, ds)

    # Create time steps k*dt
    # k goes from 0,1,...,10; # t goes from 1,0.9,...0
    # t = np.arange(T+dt, 0, -dt)
    t= T-np.arange(NTS*dt,-dt,-dt)[::-1]

    # Initialize the grid with zeros
    grid = np.zeros((len(s), len(t)))

    # Subsume the grid points into a dataframe
    # with asset price as index and time steps as columns
    grid = pd.DataFrame(grid, index=s, columns=np.around(t, 3))

    # Set Payoff at Expiration
    flag = 1 if call else -1
    grid.iloc[:,-1] = np.maximum(flag * (s - E), 0)

    # Store payoff for early exercise
    p = grid.iloc[:, -1].copy()

    # Fill the grid
    for k in range(len(t)-2, -1, -1):  # Backward in time
        for i in range(1, len(s)-1):
            delta = (grid.iloc[i+1, k+1] - grid.iloc[i-1, k+1]) / (2*ds)
            gamma = (grid.iloc[i+1, k+1] - 2*grid.iloc[i, k+1] + grid.iloc[i-1, k+1]) / (ds**2)
            theta = (-0.5 * vol**2 * s[i]**2 * gamma) - (r*s[i]*delta) + (r*grid.iloc[i, k+1])

            grid.iloc[i, k] = grid.iloc[i, k+1] - dt*theta

        # Set boundary condition at S = 0
        grid.iloc[0, k] = grid.iloc[0, k+1] * np.exp(-r*dt)

        # Set boundary condition at S = infinity
        grid.iloc[-1, k] = 2*grid.iloc[-2, k] - grid.iloc[-3, k]

        # Check for early exercise
        if american:
            intrinsic_value = np.maximum(flag * (s - E), 0) # Compute intrinsic value at t_k
            grid.iloc[:, k] = np.maximum(grid.iloc[:, k], intrinsic_value)

    # round grid values to 2 decimals
    return np.around(grid,2)


In [ ]:
# Call the function to price options
fdm_grid = efdmoption(call=False, american=False)
fdm_grid

**Visualize the payoff**

In [ ]:
# Plot Option Payoff - you need latest version of quantmod for surface plot
fdm_grid.iplot(kind = 'surface', title='Option values by Explicit FDM', xaxis_title='Maturity', yaxis_title='Spot', zaxis_title='Option Value')

## Section 3: Bilinear Interpolation

We have generated the grid and filled it with the possible option values. However, if we have to estimate option value or its derivatives on the mesh points, how can we estimate the value at points in between? The simplest way is to do a two-dimensional interpolation method called **Bilinear Interpolation**.

In [ ]:
def bilinear_interpolation(asset_price, ttm, df):

    # Find relevant rows and columns
    col1 = df.columns[df.columns >= ttm][-1]
    col2 = df.columns[df.columns < ttm][0]
    row1 = df.index[df.index <= asset_price][-1]
    row2 = df.index[df.index > asset_price][0]

    # Define points and areas, getting nearest option values
    V = [df.loc[row1, col1], df.loc[row1, col2],
           df.loc[row2, col2], df.loc[row2, col1]]

    A = [(row2 - asset_price) * (col2 - ttm),
           (row2 - asset_price) * (ttm - col1),
           (asset_price - row1) * (ttm - col1),
           (asset_price - row1) * (col2 - ttm)]

    # Interpolate values
    return np.sum(np.array(V)*np.array(A))/sum(np.array(A))

In [ ]:
# Option value, approximated
bilinear_interpolation(90,0.25,fdm_grid)

In [ ]:
# Option value, approximated
bilinear_interpolation(115,0.25,fdm_grid)

## Section 4: Convergence Analysis
Let's now compare option pricing for various asset steps (NAS) with black scholes price.

In [ ]:
# Iterate over asset steps (NAS)
nas_list = [10,20,30,40,50,60,100]

fdmoption = []
for i in nas_list:
    fdmoption.append(efdmoption(100,0.2,0.05,1,i).loc[100,1])

fdmoption

In [ ]:
# Fetch option models using quantmod convenience factory
from quantmod.models import price_option, OptionInputs, BlackScholes

bsoption = BlackScholes(
    OptionInputs(
        spot=100,
        strike=100,
        rate=0.05,
        ttm=1.0,
        volatility=0.20
    )).put_price

bsoption = bsoption.repeat(len(nas_list))

# Range of option price
bsoption

In [ ]:
# Subsume into dataframe
df = pd.DataFrame(list(zip(bsoption,fdmoption)), columns=['BS', 'FDM'], index=nas_list)
df['dev'] = df['FDM'] - df['BS']
df['% dev'] = round(df['dev'] / df['BS'] * 100.,2)

# Output
print("BS - FDM Convergence over NAS")
df

## Section 5: Also Read

📌 [Quantmod](https://docs.kannansingaravelu.com/models/)

---
[Kannan Singaravelu](https://www.linkedin.com/in/kannansi)